In [ ]:
"""
create_drive_folder.py

Creates a new dated folder in Google Drive at the start of each year.
Requires GOOGLE_CREDENTIALS_JSON to be set as a GitHub Actions secret.
"""

import os
import json
from datetime import datetime
from google.oauth2 import service_account
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/drive"]


def get_drive_service():
    """Authenticate using a service account credentials JSON stored as an env variable."""
    credentials_json = os.environ.get("GOOGLE_CREDENTIALS_JSON")
    if not credentials_json:
        raise EnvironmentError("GOOGLE_CREDENTIALS_JSON environment variable is not set.")

    credentials_info = json.loads(credentials_json)
    credentials = service_account.Credentials.from_service_account_info(
        credentials_info, scopes=SCOPES
    )
    return build("drive", "v3", credentials=credentials)


def create_folder(service, folder_name: str, parent_folder_id: str = None) -> str:
    """
    Create a folder in Google Drive.

    Args:
        service: Authenticated Google Drive service instance.
        folder_name: Name of the folder to create.
        parent_folder_id: Optional ID of a parent folder. If None, creates in root.

    Returns:
        The ID of the newly created folder.
    """
    metadata = {
        "name": folder_name,
        "mimeType": "application/vnd.google-apps.folder",
    }

    if parent_folder_id:
        metadata["parents"] = [parent_folder_id]

    folder = service.files().create(body=metadata, fields="id, name").execute()
    return folder["id"]

In [ ]:
def create_folder(service, folder_name: str, parent_folder_id: str = None) -> str:
    """
    Create a folder in Google Drive.

    Args:
        service: Authenticated Google Drive service instance.
        folder_name: Name of the folder to create.
        parent_folder_id: Optional ID of a parent folder. If None, creates in root.

    Returns:
        The ID of the newly created folder.
    """
    metadata = {
        "name": folder_name,
        "mimeType": "application/vnd.google-apps.folder",
    }

    if parent_folder_id:
        metadata["parents"] = [parent_folder_id]

    folder = service.files().create(body=metadata, fields="id, name").execute()
    return folder["id"]

In [ ]:
def main():
    year = datetime.now().year
    folder_name = f"Spotify Data {year}"

    # Optional: set a parent folder ID via env variable, or leave blank for Drive root
    parent_folder_id = os.environ.get("GOOGLE_DRIVE_PARENT_FOLDER_ID", None)

    print(f"Authenticating with Google Drive...")
    service = get_drive_service()

    print(f"Creating folder: '{folder_name}'...")
    folder_id = create_folder(service, folder_name, parent_folder_id)

    print(f"✅ Folder '{folder_name}' created successfully.")
    print(f"   Folder ID: {folder_id}")

In [ ]:
if __name__ == "__main__":
    main()